# Publications markdown generator for academicpages

**USE ONLY WHEN ADDING NEW CITATIONS, TO BE ABLE TO KEEP THE ONES ALREADY ONLINE**



Takes a TSV of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases of citations, rather than Stuart's non-standard TSV format and citation style.


## Data format

The TSV needs to have the following columns: pub_date, title, venue, excerpt, citation, site_url, and paper_url, with a header at the top. 

- `excerpt` and `paper_url` can be blank, but the others must have values. 
- `pub_date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/publications/YYYY-MM-DD-[url_slug]`

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [567]:
!cat publications.tsv

pub_date	title	venue	excerpt	citation	url_slug	paper_url	slides_url
2024-12-12	Motor Control Processes Moderate Visual Working Memory Gating	The Journal of Neuroscience	"Gating processes that regulate sensory input into visual working memory (WM) and the execution of planned actions share neural mechanisms, suggesting a mutual interaction. In a preregistered study (Open Science Framework), we examined how this interaction may result in sensory interference during WM storage using a delayed match-to-sample task. Participants (12 males, 20 females) memorized the color of a target stimulus for later report on a color wheel. The shape of the target indicated which hand they would adjust the color wheel with. During the retention interval, an interference task was presented, requiring a response with either the same or different hand as the main task. In half of the interference trials, the interfering task cue was also colored to introduce visual interference. EEG results showed early moto

## Import pandas

We are using the very handy pandas library for dataframes.

In [568]:
import pandas as pd

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [569]:
publications = pd.read_csv("publications.tsv", sep="\t", header=0, encoding="iso-8859-9")
publications


,pub_date,title,venue,excerpt,citation,url_slug,paper_url,slides_url
0,2024-12-12,Motor Control Processes Moderate Visual Workin...,The Journal of Neuroscience,Gating processes that regulate sensory input i...,"zdemir, _., Gunseli, E., & Schneider, D. (202...",Motor Control Processes Moderate Visual Workin...,http://academicpages.github.io/files/paper1.pdf,https://www.jneurosci.org/content/45/47/e06732...
1,2025-12-10,Paper Title Number 2,Journal 1,This paper is about the number 2. The number 3...,"Your Name, You. (2010). ""Paper Title Number 2....",paper-title-number-2,http://academicpages.github.io/files/paper2.pdf,sssssss
2,2025-12-31,Paper Title Number 3,Journal 1,This paper is about the number 3. The number 4...,"Your Name, You. (2015). ""Paper Title Number 3....",paper-title-number-3,http://academicpages.github.io/files/paper3.pdf,sssssss


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [570]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;",
    "ş": "&#351;",
    "ğ": "&#287;",
    "ı": "&#305;",
    "ü": "&uuml;",
    "ö": "&ouml;",
    "ç": "&ccedil;",
    "Ş": "&#350;",
    "Ğ": "&#286;",
    "İ": "&#304;",
    "Ü": "&Uuml;",
    "Ö": "&Ouml;",
    "Ç": "&Ccedil;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [571]:
import os
for row, item in publications.iterrows():
    
    md_filename = str(item.pub_date) + "-" + item.url_slug + ".md"
    html_filename = str(item.pub_date) + "-" + item.url_slug
    year = str(item.pub_date)
    
    ## YAML variables
    
    md = "---\ntitle: \""   + item.title + '"\n'
    
    md += """collection: publications""" 
       
    md += "\ncategory: \""   + str(item.pub_date[:4])  + '"\n'

    md += """permalink: /publication/""" + html_filename
 
    
    if len(str(item.excerpt)) > 5:
        md += "\nexcerpt: '" + html_escape(item.excerpt) + "'"
    
    md += "\ndate: " + item.pub_date 
    
    md += "\nvenue: '" + html_escape(item.venue) + "'"
    
    if len(str(item.slides_url)) > 5:
        md += "\nslidesurl: '" + item.slides_url + "'"

    if len(str(item.paper_url)) > 5:
        md += "\npaperurl: '" + item.paper_url + "'"
    
    md += "\ncitation: '" + html_escape(item.citation) + "'"
    
    md += "\n---"
    
    ## Markdown description for individual page
        
    if len(str(item.excerpt)) > 5:
        md += "\n**Abstract**\n"
        md += "\n" + html_escape(item.excerpt) + "\n"

    if len(str(item.slides_url)) > 5:
        md += "\n[Link](" + item.slides_url + ")\n" 

    if len(str(item.paper_url)) > 5:
        md += "\n[Download paper here](" + item.paper_url + ")\n" 
        
    md += "\nRecommended citation: \n\n"  + item.citation
    
    md_filename = os.path.basename(md_filename)
       
    with open("../_publications/" + md_filename, 'w') as f:
        f.write(md)

These files are in the publications directory, one directory below where we're working from.

In [572]:
!ls ../_publications/

2009-10-01-paper-title-number-1.md
2010-10-01-paper-title-number-2.md
2015-10-01-paper-title-number-3.md
2024-02-17-paper-title-number-4.md
2024-12-12-Motor Control Processes Moderate Visual Working Memory Gating.md
2024-12-12-paper-title-number-1.md
2025-12-10-paper-title-number-2.md
2025-12-31-paper-title-number-3.md


In [573]:
!cat ../_publications/2024-10-01-paper-title-number-1.md

cat: ../_publications/2024-10-01-paper-title-number-1.md: No such file or directory
